# Homework 3 - The model

The four questions of Module 3: the feature set and the week-of-month dummies, the
hand-written decision rules, the number of records that are predicted correctly, and
the depth of the decision tree.

**The outcome**

| # | Question | Answer |
|---|---|---|
| 1 | What is the smallest absolute correlation between a feature and the target? | **0.025** (`October_w4`) |
| 2 | What is the highest precision of the hand-written rules? | **0.588** (`pred3_manual_dgs10_5`) |
| 3 | How many test records are predicted correctly by the model alone? | **1171** |
| 4 | What is the best maximum depth of the decision tree? | **4** (precision 0.6292) |

One detail decides question 3: the `Month` column has to be encoded as the **name** of
the month and not as its number. Both encodings are legitimate, but they change the
order of the one-hot columns, and that order decides how the tree breaks a tie between
two features with the same split. With the names the count is 1171; with the numbers it
is 1187.

In [1]:
# Colab: uncomment and run the next line first.
# !pip install -q pandas numpy pyarrow scikit-learn yfinance curl_cffi
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def repo_root(start: Path) -> Path:
    """Walk up the tree until the repository layout is recognised."""
    for path in [start, *start.parents]:
        if (path / "requirements.txt").exists() and (path / "scripts").exists():
            return path
    return start


REPO = repo_root(Path.cwd().resolve())
HW = REPO / "homework3"
sys.path.insert(0, str(HW / "src"))

print("repository :", REPO.name)
print("python     :", sys.version.split()[0])

repository : repo
python     : 3.11.2


In [2]:
# The homework datasets are not in the repository. They are fetched from the course's
# Google Drive links by scripts/download_data.py, which is resumable and skips the
# files that are already there.
import subprocess

subprocess.run([sys.executable, str(REPO / "scripts" / "download_data.py")], check=False)

for name in ("stocks_df_combined_2026_09_18.parquet.brotli", "data_hw2.parquet"):
    path = REPO / "data" / name
    print(f"{'ok ' if path.exists() else 'MISSING'} {path.name}")

skipping stocks_df_combined_2026_09_18.parquet.brotli (already present)
skipping data_hw2.parquet (already present)
ok  stocks_df_combined_2026_09_18.parquet.brotli
ok  data_hw2.parquet


## The panel and the split

The shared parquet file is loaded and the split is temporal: the first 70% of the dates
are the training set, the next 15% the validation set and the last 15% the test set.
There is no shuffling anywhere, because a shuffled split leaks the future into the
training data and the test score stops meaning anything.

In [3]:
import hw3_questions

df_with_dummies, NUMERICAL, DUMMIES, TO_PREDICT = hw3_questions.build_dataframe()
print(f"rows={len(df_with_dummies)} | NUMERICAL={len(NUMERICAL)} | DUMMIES={len(DUMMIES)}")
print(df_with_dummies.groupby("split")["Date"].agg(["min", "max", "count"]).to_string())

rows=202292 | NUMERICAL=184 | DUMMIES=115
                  min        max   count
split                                   
test       2022-09-16 2026-09-18   33163
train      2000-01-03 2018-09-13  136842
validation 2018-09-14 2022-09-15   32287


## Question 1 - The dummy columns and the correlations

The week-of-month dummies are added to the other families, and every feature is
correlated with the target. The question asks for the smallest absolute correlation,
which is the feature that carries the least information about the target, and it is a
week-of-month dummy.

In [4]:
hw3_questions.question_1(df_with_dummies, NUMERICAL, DUMMIES, TO_PREDICT)

Q1 total dummies: 115 | month_wom dummies: 60
Q1 features (NUMERICAL+DUMMIES): 299
                           corr  abs_corr
month_wom_October_w4   0.024584  0.024584
month_wom_November_w3  0.023251  0.023251
month_wom_February_w1 -0.020861  0.020861
month_wom_January_w2  -0.019846  0.019846
month_wom_January_w5  -0.019792  0.019792
month_wom_November_w2  0.019103  0.019103
month_wom_January_w3  -0.018537  0.018537
month_wom_February_w2 -0.018340  0.018340
Q1 max |corr| = 0.02458 -> rounded 0.025


## Question 2 - The hand-written rules

The rules of the lecture are written out by hand: each one is a condition on a feature,
and the precision of each condition is measured on the validation slice. The best of
them is the answer.

In [5]:
hw3_questions.question_2(hw3_questions.add_manual_predictions(df_with_dummies.copy()))

Q2 pred3_manual_dgs10_5 -> flagged: 15910 precision: 0.587744
Q2 pred4_manual_dgs10_fedfunds -> flagged: 15921 precision: 0.502544
Q2 best: pred3_manual_dgs10_5 = 0.588


## Questions 3 and 4 - The model and the tree

A decision tree is fitted on the training slice and scored on the test slice. The
question asks how many test records the model gets right on its own, without the
manual rules, and the second question sweeps the maximum depth and keeps the depth with
the best precision.

In [6]:
hw3_questions.question_3_and_4(df_with_dummies, NUMERICAL, DUMMIES, TO_PREDICT)

Q3 TEST rows: 33163 | only pred5 correct (TEST): 1171 | generic fn: 1171 | all splits: 5227
Q4 precision by depth: {1: 0.5701, 2: 0.6225, 3: 0.572, 4: 0.6292, 5: 0.5991, 6: 0.5966, 7: 0.6229, 8: 0.6194, 9: 0.5956, 10: 0.5993, 11: 0.598, 12: 0.5837}
Q4 best depth: 4 precision: 0.6292

Tree head (depth 3):
 |--- cpi_core_yoy <= 0.02
|   |--- DGS10 <= 1.75
|   |   |--- DGS1 <= 0.56
|   |   |   |--- growth_snp500_90d <= 1.14
|   |   |   |   |--- truncated branch of depth 7
|   |   |   |--- growth_snp500_90d >  1.14
|   |   |   |   |--- truncated branch of depth 7
|   |   |--- DGS1 >  0.56
|   |   |   |--- growth_btc_usd_30d <= 1.07
|   |   |   |   |--- truncated branch of depth 7
|   |   |   |--- growth_btc_usd_30d >  1.07
|   |   |   |   |--- truncated branch of depth 7
|   |--- DGS10 >  1.75
|   |   |--- growth_wti_oil_365d <= 1.50
|   |   |   |--- Month_July <= 0.50
|   |   |   |   |--- truncated branch of depth 7
|   |   |   |--- Month_July >  0.50
|   |   |   |   |--- truncated branch

## The answers

In [7]:
# The answers, and the file the scripts write. The notebook and the script produce the
# same JSON, so the answer in the README, the answer in the file and the answer printed
# here cannot drift apart.
import json

answers = hw3_questions.answers
out = HW / "output" / "hw3_answers.json"
out.write_text(json.dumps(answers, indent=2, default=str))

for key, value in answers.items():
    print("=" * 70)
    print(key.upper(), "-", value.get("question", ""))
    for field, item in value.items():
        if field == "question":
            continue
        if isinstance(item, (dict, list)) and len(str(item)) > 200:
            print("  " + field + ": <" + type(item).__name__ + " of " + str(len(item)) + ">")
        else:
            print("  " + field + ": " + str(item))

print()
print("Saved:", out.relative_to(REPO))

Q1 - What is the ABSOLUTE CORRELATION VALUE of the most correlated dummy variable _w<week_of_month> with the binary outcome is_positive_growth_30d_future?
  n_dummies_total: 115
  n_month_wom_dummies: 60
  n_features_total: 299
  top10: <list of 10>
  max_abs_corr: 0.02458416756158805
  max_abs_corr_rounded: 0.025
  top_dummy: month_wom_October_w4
Q2 - What is the precision score for the best of the NEW predictions (pred3 or pred4), rounded to 3 digits after the comma?
  per_rule: <dict of 2>
  best_rule: pred3_manual_dgs10_5
  best_precision: 0.5877435575109994
  best_precision_rounded: 0.588
Q3 - What is the total number of records in the TEST dataset where the new prediction pred5_clf_10 is correct, while all 'hand' rule predictions (pred0 to pred4) are incorrect?
  n_test_records: 33163
  only_pred5_is_correct_test: 1171
  only_pred5_is_correct_all_splits: 5227
  generic_function_result_test: 1171
Q4 - What is the optimal tree depth (from 1 to 12) for a DecisionTreeClassifier?
  pr